# Translation quality control: recorded pipeline

**Automatic speech recognition (ASR)** turns English audio into English text;
**machine translation (MT)** translates that text into German. This sequential
**cascade** can make errors in either stage. Here, **quality control (QC)** asks whether
the German preserves the meaning of the recognized English, even when that English is wrong.

A **chunk** is a 4- or 8-second audio processing block. A **reference segment** is a
supplied scoring unit. A **QC window** is a 24-second passage containing several
chunks, inspected with neighboring context. A **reference** is supplied comparison
text; a **prediction** (or hypothesis) is system output.

Two model reviewers have different roles:

- **Jev**, Typesafe's `typesafe/jev-1.13-20260917`, is the first screen. It assigns category scores to flag passages for inspection.
- **Claude Sonnet 4.5**, Anthropic's `anthropic/claude-sonnet-4.5`, is the second, detailed reviewer. It gives category verdicts and quotes evidence for selected passages. It is a model, not a human reviewer.

This notebook was prepared after measurement and reads saved inputs, requests,
responses and summaries. **All cells are offline and read-only.** The underlying
QC prompts and thresholds remain unchanged; full execution histories are archived separately.

In [1]:
from pathlib import Path
import ast
import json
import sys
from IPython.display import Code, JSON, Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "CODE_GUIDE.md").is_file() and (p / "quality-control").is_dir())
sys.path.insert(0, str(ROOT / "quality-control"))
import qc_notebook as qc

def read(path):
    return json.loads(path.read_text(encoding="utf-8"))

def read_lines(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

def show_source(relative_path, name):
    text = (ROOT / relative_path).read_text(encoding="utf-8")
    node = next(n for n in ast.walk(ast.parse(text))
                if isinstance(n, (ast.FunctionDef, ast.ClassDef)) and n.name == name)
    display(Code("\n".join(text.splitlines()[node.lineno-1:node.end_lineno]), language="python"))

print("Loaded local submission:", ROOT.name)


Loaded local submission: TinyML-code-submission


In [2]:
QC = ROOT / "quality-control"
DATA = QC / "recorded/main-api-v2"
REVIEW = DATA / "api-review-v1"
manifest = read(DATA / "manifest.json")
sample = read(DATA / "sample_manifest.json")
windows = read_lines(DATA / "windows.jsonl")
review_manifest = read(REVIEW / "review_manifest.json")


## 1. Recognition error analysis

**Word error rate (WER)** compares recognized English with normalized English
references: (substitutions + deletions + insertions) / reference-word count. Lower is
better. **One substitution, one deletion and one insertion against ten reference
words gives 30% WER.**

A deletion in this word alignment is not automatically a meaningful translation
omission. The former counts recognition differences; the latter asks whether German
loses information from its English input. The MT fidelity judgments below use the
actual recognized English, without assuming it is correct.

In [3]:
qc.show_asr(DATA)


Configuration,Reference words,Substitutions,Deletions,Insertions,WER %
laptop/base.en/4s,4686,767,178,318,26.95
laptop/base.en/8s,4686,626,186,565,29.39
laptop/tiny.en/4s,4686,955,192,770,40.91
laptop/tiny.en/8s,4686,759,184,284,26.18
pi/base.en/4s,4686,779,161,295,26.36
pi/base.en/8s,4686,635,165,274,22.92
pi/tiny.en/4s,4686,926,194,568,36.02
pi/tiny.en/8s,4686,731,180,289,25.61


Compare the error types as well as total WER. Laptop/base.en has higher WER at 8 seconds than at 4 seconds, illustrating why longer chunks do not guarantee better recognition.

## 2. Fixed sample and targeted diagnostics

The preparation audit covered 96 main jobs. One of each text-identical repetition was retained, producing 696 windows from 48 outputs. Two shared time passages per talk, selected with seed 20261022, yielded 96 configuration outputs. Six diagnostics were selected separately; they are not added to a random-sample rate.


In [4]:
qc.table([{k: manifest[k] for k in ("audited_runs", "retained_runs", "window_count", "selected_count", "diagnostic_count")}])
display(Markdown("Removing repeated identical text halves the 96 runs to 48 retained outputs; only 96 of the 696 available windows enter the fixed sample, with six separate diagnostics."))
qc.table({"Talk": r["talk_id"], "Available passages": r["population"],
          "Selected passages": "; ".join(r["selected_passage_ids"])} for r in sample["selection"])

audited_runs,retained_runs,window_count,selected_count,diagnostic_count
96,48,696,96,6


Removing repeated identical text halves the 96 runs to 48 retained outputs; only 96 of the 696 available windows enter the fixed sample, with six separate diagnostics.

Talk,Available passages,Selected passages
ERmKpJPPDc,15,ERmKpJPPDc/w009/216.000000-240.000000; ERmKpJPPDc/w012/288.000000-312.000000
EqmWoxNDIr,13,EqmWoxNDIr/w002/48.000000-72.000000; EqmWoxNDIr/w011/264.000000-288.000000
UOlPKyCVgg,13,UOlPKyCVgg/w011/264.000000-288.000000; UOlPKyCVgg/w012/288.000000-299.178688
krJSAnVcGR,13,krJSAnVcGR/w007/168.000000-192.000000; krJSAnVcGR/w012/288.000000-304.126688
rxrToXvRyM,18,rxrToXvRyM/w000/0.000000-24.000000; rxrToXvRyM/w015/360.000000-384.000000
xhjBHGVOyQ,15,xhjBHGVOyQ/w002/48.000000-72.000000; xhjBHGVOyQ/w012/288.000000-312.000000


The same two time passages per talk are shared across configurations, making their English/German outputs comparable. These repeated views of a passage are not independent content samples.

## 3. Inspect the English ASR and German MT

Each focal passage includes neighboring context. Each row below pairs the English
and German for one **chunk ID**, with its audio interval. These IDs are the
`chunk_id` values used in reviewer quotations, so evidence can be located precisely.
“Before” and “After” provide context; “Focal” is the passage being judged.

This is one saved selected case, also discussed as Case 3 in the author's note.
Change `WINDOW_ID` to another ID from the saved selection to inspect it.

In [5]:
WINDOW_ID = "UOlPKyCVgg/laptop/UOlPKyCVgg-base-4s-r1/w011/264.000000-288.000000"
qc.show_window(DATA, WINDOW_ID)


Chunk ID,Audio interval (s),English ASR,German MT
64,256.000–260.000,from stability issues.,von Stabilitätsfragen.
65,260.000–264.000,our proper system of a better performance.,Unser richtiges System von besserer Leistung.
Chunk ID,Audio interval (s),English ASR,German MT
66,264.000–268.000,9 of the 11 don't use the task. I'm supposed to globally use the res...,9 der 11 benutzen die Aufgabe nicht.
67,268.000–272.000,"of the generic model here, come on back.","des generischen Modells hier, komm zurück."
68,272.000–276.000,Also observing that the special data is the,"Auch zu beobachten, dass die besonderen Daten sind die"
69,276.000–280.000,Most research data is better but it doesn't scale well.,"Die meisten Forschungsdaten sind besser, aber sie skalieren nicht gut."
70,280.000–284.000,All the pre-trained model obtained from Nachos are freely available.,Alle von Nachos erhaltenen vortrainierten Modelle sind frei erhältlich.
71,284.000–288.000,and on the eugene face and all the training strips are on the...,und auf dem eugenen Gesicht und alle Trainingsstreifen sind auf dem...
Chunk ID,Audio interval (s),English ASR,German MT


In focal chunk 66, the English continues after “9 of the 11” but the German ends after its first sentence. The later reviewer examines that fragment; locating it does not by itself establish that its meaning is recoverable.

## 4. Exact Jev requests and saved decisions

Jev screens for meaningful omission, meaning change/negation, entity/quantity/term error, and unsupported addition. A threshold creates inspection candidates; it is not a validated error probability. The displayed payload is from a saved decision record. The saved decision includes the exact request and response; full request histories are archived separately.


In [6]:
decisions = read_lines(DATA / "jev_decisions.jsonl")
decision = next(d for d in decisions if d["window_id"] == WINDOW_ID and d["status"] == "ok")
display(JSON(decision["request"], expanded=False))
qc.table([{"Model": decision["model"], "Status": decision["status"],
           "HTTP status": decision["http_status"], "Scores": decision["probabilities"]}])


<IPython.core.display.JSON object>

Model,Status,HTTP status,Scores
typesafe/jev-1.13-20260917,ok,200,"{'entity_or_quantity_error': 0.51, 'meaning_change_or_negation': 0.27, 'meaningful_omission': 0.47, 'unsupported_addition': 0.23}"


The category scores belong to this one window. Compare each score with the 0.5 threshold to see what Jev flags; a successful HTTP status only confirms that the request returned successfully.

## 5. Threshold sensitivity

The saved report separates the fixed sample from diagnostics and shows threshold sensitivity at 0.5, 0.7 and 0.9. Flag counts describe this model's screening behavior; they are not human-validated error rates. Zero omission flags does not establish that no omissions occurred.


In [7]:
qc_summary = read(DATA / "summary.json")
def fraction(rate):
    return f"{rate['numerator']}/{rate['denominator']}" if rate["denominator"] else "Not judged"

qc.table({"Configuration": name, "Sample windows": row["windows"], "Judged": row["successful"],
          "Failed": row["failed"],
          "Omission flags / judged": fraction(row["thresholds"]["0.5"]["suggested_categories"]["meaningful_omission"]),
          "Any flags / judged": fraction(row["thresholds"]["0.5"]["candidate_review_rate"])}
         for name, row in qc_summary["sample_by_configuration"].items())
display(Markdown("All configurations have zero omission flags at 0.5, while other categories produce candidates. This is Jev's screening pattern, not evidence that meaningful omissions never occur."))
qc.table({"Selection": label, "Threshold": threshold,
          "Any flags / judged": fraction(row["candidate_review_rate"])}
         for label, subset in (("Fixed sample", qc_summary["review_subset"]),
                               ("Diagnostics", qc_summary["diagnostic_subset"]))
         for threshold, row in subset["thresholds"].items())
display(Markdown("In the fixed sample, raising the threshold from 0.5 to 0.7 to 0.9 reduces candidates from 81/96 to 52/96 to 0/96. The six targeted diagnostics have a separate denominator."))

Configuration,Sample windows,Judged,Failed,Omission flags / judged,Any flags / judged
laptop/base.en/4s,12,12,0,0/12,10/12
laptop/base.en/8s,12,12,0,0/12,11/12
laptop/tiny.en/4s,12,12,0,0/12,11/12
laptop/tiny.en/8s,12,12,0,0/12,10/12
pi/base.en/4s,12,12,0,0/12,10/12
pi/base.en/8s,12,12,0,0/12,10/12
pi/tiny.en/4s,12,12,0,0/12,10/12
pi/tiny.en/8s,12,12,0,0/12,9/12


All configurations have zero omission flags at 0.5, while other categories produce candidates. This is Jev's screening pattern, not evidence that meaningful omissions never occur.

Selection,Threshold,Any flags / judged
Fixed sample,0.5,81/96
Fixed sample,0.7,52/96
Fixed sample,0.9,0/96
Diagnostics,0.5,4/6
Diagnostics,0.7,1/6
Diagnostics,0.9,0/6


In the fixed sample, raising the threshold from 0.5 to 0.7 to 0.9 reduces candidates from 81/96 to 52/96 to 0/96. The six targeted diagnostics have a separate denominator.

## 6. Detailed API review

Claude Sonnet 4.5 is the detailed reviewer in this stage. The follow-up was designed after examining Jev outputs. It selected 16 high-score, eight borderline, six unflagged and six diagnostic cases. The reviewer saw the text and rubric, without Jev scores in its API payload. Local selection metadata still records those scores for later comparison.

The four categories have independent verdicts and quoted evidence. The actual ASR is the source: faithfully carried recognition errors are not automatically additional MT errors. The original prompt and response schema below are preserved.

In [8]:
requests = read_lines(REVIEW / "review_requests.jsonl")
display(JSON(review_manifest["settings"], expanded=True))
display(JSON(requests[0]["payload"], expanded=False))


<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

## 7. Successful and failed attempts

HTTP success and locally validated review success are different. Responses can fail because evidence quotations do not occur in the supplied text or a category requires missing evidence. Authentication failures are also retained. The four failed attempts below concern three windows; one window was tried twice. Final outcomes cover 33 successes and three failed cases, from 37 attempts in total. No failure has been silently converted into an accepted judgment.


In [9]:
completed = read_lines(REVIEW / "review_outcomes.jsonl")
failures = read_lines(REVIEW / "failed_review_examples.jsonl")
qc.table({"Window": a["window_id"], "Status": a["status"],
          "HTTP": a.get("http_status"), "Error": a.get("error", "")} for a in failures)
display(Markdown("The repeated window ID shows why four failed attempts correspond to three failed cases. Compare HTTP status with the local error: a successful HTTP response can still fail evidence validation."))
selected_response = next(a for a in completed if a["status"] == "ok" and a["window_id"] == WINDOW_ID)
display(JSON(selected_response["annotation"], expanded=True))

Window,Status,HTTP,Error
ERmKpJPPDc/laptop/ERmKpJPPDc-tiny-4s-r1/w005/120.000000-144.000000,failed,200,ValueError: Meaning/term error requires English and German evidence
ERmKpJPPDc/laptop/ERmKpJPPDc-tiny-4s-r1/w005/120.000000-144.000000,failed,200,ValueError: Meaning/term error requires English and German evidence
EqmWoxNDIr/pi/EqmWoxNDIr-base-4s-r1/w002/48.000000-72.000000,failed,401,ValueError: API returned HTTP 401 or an error object; see saved response
EqmWoxNDIr/laptop/EqmWoxNDIr-base-4s-r1/w002/48.000000-72.000000,failed,200,ValueError: Evidence is not an exact target substring


The repeated window ID shows why four failed attempts correspond to three failed cases. Compare HTTP status with the local error: a successful HTTP response can still fail evidence validation.

<IPython.core.display.JSON object>

## 8. Evaluator comparisons and author inspection

The final report has 33 successful windows out of 36 selected. Counts are model
agreement on selected cases, not accuracy, recall or population error prevalence.
The author inspection is separate and does not provide expert German fluency labels.

The comparison uses a **Jev score ≥ 0.5** as a flag in each category:

| Label | Meaning |
|---|---|
| `both_flag` | Jev flags the category and Claude marks it `present`. |
| `jev_only` | Jev flags it and Claude marks it `absent`. |
| `reviewer_only` | Jev does not flag it and Claude marks it `present`. |
| `both_unflagged` | Jev does not flag it and Claude marks it `absent`. |
| `uncertain` | Claude cannot decide, regardless of Jev's score. |
| `not_reviewed` / `failed_or_interrupted` | No accepted detailed judgment is available. |

Each row below counts one category within one selection group. A window appears in
all four categories, so summing every row would count each window repeatedly.

In [10]:
result = read(REVIEW / "review_summary.json")
qc.table([{k: result[k] for k in ("selected_windows", "successful_windows", "interpretation")}])
display(Markdown("Three of the 36 selected windows lack an accepted detailed review; the comparison must retain that missing evidence."))
qc.table(result["by_group_and_category"])

selected_windows,successful_windows,interpretation
36,33,"Selected-case model agreement, not accuracy, recall or population error rates"


Three of the 36 selected windows lack an accepted detailed review; the comparison must retain that missing evidence.

group,category,selected,both_flag,both_unflagged,jev_only,reviewer_only,uncertain,not_reviewed,failed_or_interrupted
high,meaningful_omission,16,0,14,0,0,0,0,2
high,meaning_change_or_negation,16,3,6,2,0,3,0,2
high,entity_or_quantity_error,16,11,0,3,0,0,0,2
high,unsupported_addition,16,0,13,1,0,0,0,2
borderline,meaningful_omission,8,0,5,0,3,0,0,0
borderline,meaning_change_or_negation,8,3,3,1,1,0,0,0
borderline,entity_or_quantity_error,8,4,1,3,0,0,0,0
borderline,unsupported_addition,8,0,8,0,0,0,0,0
unflagged,meaningful_omission,6,0,6,0,0,0,0,0
unflagged,meaning_change_or_negation,6,0,4,0,2,0,0,0


The borderline group contains three `reviewer_only` omission judgments, despite no Jev omission flags. This identifies disagreements worth inspecting; it does not establish that Claude is correct.

### Locate the disputed evidence in Case 1

The original author note below joins chunks into paragraphs. Use this display to
locate its quotations first. The stored window ID begins `krJSAnVcGR`; the note's
opening label accidentally drops the initial `k`.

In [11]:
CASE_1_WINDOW_ID = "krJSAnVcGR/pi/krJSAnVcGR-base-4s-r1/w007/168.000000-192.000000"
qc.show_window(DATA, CASE_1_WINDOW_ID)

Chunk ID,Audio interval (s),English ASR,German MT
40,160.000–164.000,My background knowledge is assumed to be available at pre-trained time.,Mein Hintergrundwissen wird zur vorschulischen Zeit als verfügbar angenommen.
41,164.000–168.000,"Second, there's a back one both setting.","Zweitens, es gibt eine hintere eine beide Einstellung."
Chunk ID,Audio interval (s),English ASR,German MT
42,168.000–172.000,"both at pre-trained time and in frequency time. Lastly, the back one.",sowohl zur vortrainierten Zeit als auch in der Frequenz. Schließlich die hintere.
43,172.000–176.000,and the current setting. We both knowledge types available only at the inference time.,und die aktuelle Einstellung. Wir beide Wissenstypen nur zur Schlussfolgerung Zeit verfügbar.
44,176.000–180.000,This last setting is especially interesting. Since it's similar...,Diese letzte Einstellung ist besonders interessant. Da es ähnlich ist...
45,180.000–184.000,the case with the background noise necessary to solve a task is not part of the pre-trained data.,"der Fall mit dem Hintergrundrauschen, der zur Lösung einer Aufgabe erforderlich ist, nicht Teil der vortrainierten Daten ist."
46,184.000–188.000,"for example, because new occupations have developed since the time of the day.","Zum Beispiel, weil sich seit der Tageszeit neue Berufe entwickelt haben."
47,188.000–192.000,pre-trained. Here's an example of how we can,"Hier ist ein Beispiel, wie wir"
Chunk ID,Audio interval (s),English ASR,German MT


**Chunk 42** contains “pre-trained” and the German “vortrainierten”. **Chunk 47**
contains a later “pre-trained” with no corresponding term in that chunk's German.
The reviewer cites chunk 47, while the author's rebuttal quotes German from chunk 42.
The chunk IDs make this distinction visible. Whether the later fragment carries
additional recoverable meaning still needs interpretation; the earlier occurrence
alone does not resolve that question.

### Original author note (historical record)

The note is reproduced unchanged, including its merged paragraphs and unsuccessful
Case 4 lookup. For a chunk-by-chunk view of any case, use `qc.show_window(DATA, id)`.
Case 4's corrected stored ID is
`ERmKpJPPDc/laptop/ERmKpJPPDc-tiny-4s-r1/w005/120.000000-144.000000`.

In [12]:
display(Markdown((QC / "recorded/manual-inspection.md").read_text(encoding="utf-8")))


## Case 1



**krJSAnVcGR/pi/krJSAnVcGR-base-4s-r1/w007/168.000000-192.000000**

|                   Category |  Jev | API reviewer |     Comparison |                                                  Explanation |                                                     Evidence |
| -------------------------: | ---: | -------: | -------------: | -----------------------------------------------------------: | -----------------------------------------------------------: |
|        meaningful_omission | 0.22 |  present |  API reviewer_only | The word 'pre-trained' at the start of chunk 47 is completely omitted from the German translation, which jumps directly to 'Hier ist ein Beispiel'. This omission loses the connection to the previous discussion about pre-training. | [{"chunk_id":47,"scope":"focal","source_quote":"pre-trained","target_quote":null}] |
| meaning_change_or_negation | 0.34 |  present |  API reviewer_only | In chunk 42, 'in frequency time' becomes 'in der Frequenz' (in the frequency), losing the temporal aspect. In chunk 43, the grammatical structure breaks down: 'We both knowledge types available' is fragmented in German as 'Wir beide Wissenstypen nur zur...verfügbar', changing the intended meaning about knowledge type availability. | [{"chunk_id":42,"scope":"focal","source_quote":"in frequency time","target_quote":"in der Frequenz"},{"chunk_id":43,"scope":"focal","source_quote":"We both knowledge types available","target_quote":"Wir beide Wissenstypen nur zur Schlussfolgerung Zeit verfügbar"}] |
|   entity_or_quantity_error |  0.5 |  present |      both_flag | The English term 'inference time' is a technical term in machine learning referring to when a model makes predictions. The German translation 'Schlussfolgerung Zeit' (conclusion time) changes the technical meaning, as the standard German term would be 'Inferenzzeit' or 'Vorhersagezeit'. | [{"chunk_id":43,"scope":"focal","source_quote":"inference time","target_quote":"Schlussfolgerung Zeit"}] |
|       unsupported_addition | 0.19 |   absent | both_unflagged | No substantive content in the focal German chunks appears unsupported by the focal or neighboring English text. The German translations, while sometimes awkward or incomplete, do not add new claims beyond what is present in the source. |                                                           [] |

#### Before

English ASR

```
My background knowledge is assumed to be available at pre-trained time. Second, there's a back one both setting.
```

German MT

```
Mein Hintergrundwissen wird zur vorschulischen Zeit als verfügbar angenommen. Zweitens, es gibt eine hintere eine beide Einstellung.
```

#### Focal

English ASR

```
both at pre-trained time and in frequency time. Lastly, the back one. and the current setting. We both knowledge types available only at the inference time. This last setting is especially interesting. Since it's similar... the case with the background noise necessary to solve a task is not part of the pre-trained data. for example, because new occupations have developed since the time of the day. pre-trained. Here's an example of how we can
```

German MT

```
sowohl zur vortrainierten Zeit als auch in der Frequenz. Schließlich die hintere. und die aktuelle Einstellung. Wir beide Wissenstypen nur zur Schlussfolgerung Zeit verfügbar. Diese letzte Einstellung ist besonders interessant. Da es ähnlich ist... der Fall mit dem Hintergrundrauschen, der zur Lösung einer Aufgabe erforderlich ist, nicht Teil der vortrainierten Daten ist. Zum Beispiel, weil sich seit der Tageszeit neue Berufe entwickelt haben. Hier ist ein Beispiel, wie wir
```

#### After

English ASR

```
control the availability of facts and true sources. setting. We assumed that to back up knowledge, potassium-seq elected seats in government
```

German MT

```
die Verfügbarkeit von Fakten und wahren Quellen zu kontrollieren. Wir nahmen an, dass, um das Wissen zu sichern, Kalium-Seq gewählt Sitze in der Regierung
```



## Manual inspection

### Question: “is pre-trained” represented anywhere in the German, including neighboring chunks?

Window ID: **krJSAnVcGR/pi/krJSAnVcGR-base-4s-r1/w007/168.000000-192.000000**
Inspector and date: F
English information being checked:
Relevant German text and chunk IDs:
Observation: found / **appears missing** / uncertain
English source clarity: clear / **partly clear** / unclear
Reason:  The earlier occurrence of “pre-trained” is represented by “vortrainierten” in chunk 42. However, the API reviewer refers to a separate occurrence at the beginning of chunk 47, whose German translation starts directly with “Hier ist ein Beispiel”. This later occurrence appears missing. Its contribution to the intended meaning remains uncertain because the surrounding English ASR output is fragmented.



__________________________________________

## Case 2



**xhjBHGVOyQ/laptop/xhjBHGVOyQ-tiny-8s-r1/w002/48.000000-72.000000**

|                   Category |  Jev | API reviewer |     Comparison |                                                  Explanation |                                                     Evidence |
| -------------------------: | ---: | -------: | -------------: | -----------------------------------------------------------: | -----------------------------------------------------------: |
|        meaningful_omission | 0.22 |  present |  API reviewer_only | The English 'multimodal protein models' (plural) is translated as 'multimodalen Proteins' (singular genitive), omitting the explicit reference to 'models' as a separate noun. The German construction suggests 'tuning of a multimodal protein' rather than 'tuning multimodal protein models'. | [{"chunk_id":6,"scope":"focal","source_quote":"models","target_quote":null}] |
| meaning_change_or_negation | 0.66 |  present |      both_flag | In chunk 6, 'instruction tuning' (a technical term for a training method) is translated as 'Anleitung zum Tuning' (guide/instruction for tuning), changing the meaning from a technical process to guidance about tuning. In chunk 8, 'instruction data set' becomes 'Befehlsdatensatz' (command dataset), which shifts the meaning from training instructions to commands. | [{"chunk_id":6,"scope":"focal","source_quote":"instruction tuning a multimodal protein models","target_quote":"Anleitung zum Tuning eines multimodalen Proteins"},{"chunk_id":8,"scope":"focal","source_quote":"instruction data set","target_quote":"Befehlsdatensatz"}] |
|   entity_or_quantity_error | 0.88 |  present |      both_flag | The English ASR contains 'consider-revolt discrepancy' which appears to be a recognition error (likely 'considerable' was intended). The German translates this as 'Diskrepanz zwischen revolt und revolt' (discrepancy between revolt and revolt), which is a different formulation that changes the structure. Additionally, 'LP' in English is preserved as 'LP' in German, but without context it's unclear if this technical abbreviation is correctly handled. | [{"chunk_id":8,"scope":"focal","source_quote":"consider-revolt discrepancy","target_quote":"Diskrepanz zwischen revolt und revolt"},{"chunk_id":8,"scope":"focal","source_quote":"LP","target_quote":"LP"}] |
|       unsupported_addition | 0.49 |   absent | both_unflagged | No substantive content in the German translation appears unsupported by the English source. The translations are attempts to render the English ASR content, even where that content contains apparent recognition errors. |                                                           [] |

#### Before

English ASR

```
are improving the serial shot performance on language only task, where computer vision and multimodal task has been left out.
```

German MT

```
Verbesserung der seriellen Schussleistung nur auf Sprachaufgabe, bei der Computersicht und multimodale Aufgabe ausgelassen wurde.
```

#### Focal

English ASR

```
Therefore, in this work, we want to investigate whether instruction tuning a multimodal protein models can actually... improve the generalization to unseen multimodal tasks. Additionally, at the time of our research. We discovered a consider-revolt discrepancy in availability of instruction data set between LP and multi-modal.
```

German MT

```
Deshalb wollen wir in dieser Arbeit untersuchen, ob eine Anleitung zum Tuning eines multimodalen Proteins tatsächlich... Verbesserung der Verallgemeinerung zu unsichtbaren multimodalen Aufgaben. Darüber hinaus zum Zeitpunkt unserer Forschung. Wir entdeckten eine Diskrepanz zwischen revolt und revolt in der Verfügbarkeit von Befehlsdatensatz zwischen LP und multimodal.
```

#### After

English ASR

```
It's a very existing model with 1600 lunged only instruction tasks. However, there is no large scale publicly available.
```

German MT

```
Es ist ein sehr existierendes Modell mit 1600 nur gelungenen Lehraufgaben. Es ist jedoch kein großes öffentliches Format verfügbar.
```

## Manual inspection

### Question: Is the reference to “models” preserved?

Window ID: **xhjBHGVOyQ/laptop/xhjBHGVOyQ-tiny-8s-r1/w002/48.000000-72.000000**
Inspector and date: **F**
English information being checked:
Relevant German text and chunk IDs:
Observation: found / **appears missing** / uncertain
English source clarity: clear / **partly clear** / unclear
Reason: The English phrase “multimodal protein models” contains a reference to models. The corresponding German phrase, “eines multimodalen Proteins”, does not preserve that reference. The English source is partly unclear, but “models” appears missing from the translation.

_________________________

## Case 3

**UOlPKyCVgg/laptop/UOlPKyCVgg-base-4s-r1/w011/264.000000-288.000000**

|                   Category |  Jev | API reviewer |     Comparison |                                                  Explanation |                                                     Evidence |
| -------------------------: | ---: | -------: | -------------: | -----------------------------------------------------------: | -----------------------------------------------------------: |
|        meaningful_omission | 0.47 |  present |  API reviewer_only | The phrase 'I'm supposed to globally use the res...' in chunk 66 is omitted from the German translation. This appears to be a substantive statement about intended usage that is not recovered in the focal or neighboring German chunks. | [{"chunk_id":66,"scope":"focal","source_quote":"I'm supposed to globally use the res...","target_quote":null}] |
| meaning_change_or_negation | 0.27 |   absent | both_unflagged | The focal chunks preserve the core propositions from English to German. While the ASR appears fragmented and incomplete, the German translation faithfully renders the available English content without changing substantive meaning or introducing negations. |                                                           [] |
|   entity_or_quantity_error | 0.51 |   absent |       jev_only | The quantity '9 of the 11' is correctly translated as '9 der 11'. No entity or quantity error is present. | [{"chunk_id":66,"scope":"focal","source_quote":"9 of the 11","target_quote":"9 der 11"}] |
|       unsupported_addition | 0.23 |   absent | both_unflagged | All content in the focal German chunks corresponds to content in the focal English chunks. No substantive assertions appear in German that lack support in the English source. |                                                           [] |

#### Before

English ASR

```
from stability issues. our proper system of a better performance.
```

German MT

```
von Stabilitätsfragen. Unser richtiges System von besserer Leistung.
```

#### Focal

English ASR

```
9 of the 11 don't use the task. I'm supposed to globally use the res... of the generic model here, come on back. Also observing that the special data is the Most research data is better but it doesn't scale well. All the pre-trained model obtained from Nachos are freely available. and on the eugene face and all the training strips are on the...
```

German MT

```
9 der 11 benutzen die Aufgabe nicht. des generischen Modells hier, komm zurück. Auch zu beobachten, dass die besonderen Daten sind die Die meisten Forschungsdaten sind besser, aber sie skalieren nicht gut. Alle von Nachos erhaltenen vortrainierten Modelle sind frei erhältlich. und auf dem eugenen Gesicht und alle Trainingsstreifen sind auf dem...
```

#### After

English ASR

```
our GitHub repository. So thank you for. for this presentation and we are looking forward to.
```

German MT

```
Also danke für unser GitHub-Repository. für diese Präsentation und wir freuen uns darauf.
```

## Manual inspection

### Question: Does the allegedly omitted fragment contain recoverable, meaningful information?

Window ID:
Inspector and date: **F**
English information being checked:
Relevant German text and chunk IDs:
Observation: found / appears missing / **uncertain**
English source clarity: clear / partly clear / **unclear**
Reason: TThe English fragment “I'm supposed to globally use the res...” is incomplete and difficult to interpret. It appears absent from the German translation, but I could not determine confidently what meaningful information was lost.
______________________________

## Case 4


Window ID: ERmKpJPPDc/laptop/ERmKpJPPDc-tiny-4s-r1/w005/120.000000-144.000000

Inspector: F

English information being checked: The percentage expression in chunk 35, beginning “30% of the total”.

Observation: uncertain

English source clarity: unclear

Reason: The English source repeats “total” more than 100 times, and the German translation similarly repeats “Gesamtsumme”. The repetition makes the intended source message unclear, so I could not confidently assess the meaningful omission.

Review status: The automated review failed validation. This manual observation is retained separately from the accepted automated results.


## 9. Trace the code and files

- `translation_qc/main_experiment.py` audits runs, constructs shared windows and selects passages.
- `translation_qc/jev.py` constructs/sends requests and preserves decisions and usage.
- `qc_review.py:select`, `request_payload`, `validate_review` and `run` implement the detailed stage.
- `recorded/main-api-v2/` contains the window/sample manifests and Jev evidence.
- `recorded/main-api-v2/api-review-v1/` contains reviewer manifests, exact payloads, responses and comparisons.

Recorded manifests retain the original computer paths. The notebook reads retained data directly; it does not resume API execution or rebuild the experimental review report. Full invocation and attempt histories are archived separately.
